# Phase 1 — Data analysis and feature engineering

Churn prevention agent for credit card customers.

**Churn here means the customer closed their credit card.** They remain a customer of the bank — they have ended this one product. So the monthly purchase and payment columns are card activity, and every offer we design later is a card retention offer.

**What this notebook does, in order:**

1. Loads the data and checks its condition
2. Repeats the quality checks we reasoned through, so they are recorded in code
3. Compares customers who closed their card against those who kept it
4. Builds the new columns
5. Checks each new column earns its place
6. Saves the result for the modelling step

Every decision is written up in `docs/PHASE1_ANALYSIS_AND_FEATURES.md`. A decision with no reason next to it is not a decision yet.

## 0. Running this in Google Colab

Run the cell below **first**. It will ask you to choose the dataset file from your computer.

If you would rather keep the file in Google Drive, comment out the upload block and uncomment the Drive block instead — that way you only upload it once.

In [ ]:
# --- Google Colab: get the data file ---
# Option A: upload it each session (simplest to start with)
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    uploaded = files.upload()               # choose bank_churn_dataset.csv
    DATA = list(uploaded.keys())[0]
else:
    DATA = "../data/raw/bank_churn_dataset.csv"   # running locally

print("using:", DATA)

# --- Option B: keep it in Google Drive instead (uncomment to use) ---
# from google.colab import drive
# drive.mount("/content/drive")
# DATA = "/content/drive/MyDrive/churn/bank_churn_dataset.csv"

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


df = pd.read_csv(DATA)
print(f"{df.shape[0]:,} customers, {df.shape[1]} columns")

We refer to these column groups constantly, so name them once.

In [ ]:
PURCHASES = [f"purchase_month_{i}" for i in range(1, 7)]
PAYMENTS  = [f"payment_month_{i}"  for i in range(1, 7)]
ANSWER    = "churned"

# Month 1 is the OLDEST month, month 6 the most recent.
# Confirmed with the manager. Every trend column below depends on this.
# If it were reversed, a collapsing customer would look like a growing one.

## 2. What condition is the data in?

Before anything else: how big is it, what types, is anything missing, is anything duplicated.

In [ ]:
print("shape:", df.shape)
print("duplicate rows:", df.duplicated().sum())
print("duplicate customer ids:", df.customer_id.duplicated().sum())
print()

summary = pd.DataFrame({
    "type": df.dtypes,
    "missing": df.isna().sum(),
    "unique": df.nunique(),
})
summary

## 3. The answer column

How lopsided is it? This one number decides which measures we are allowed to judge the model on.

In [ ]:
counts = df[ANSWER].value_counts()
share  = df[ANSWER].value_counts(normalize=True)

print(f"stayed: {counts[0]:,}  ({share[0]:.1%})")
print(f"left  : {counts[1]:,}  ({share[1]:.1%})")
print()
print(f"A model answering 'nobody ever leaves' would be right {share[0]:.0%} of the time.")
print("So accuracy is not a measure we can use.")

## 4. Quality checks

Two different problems, easy to confuse:

- **Leakage** is about *time*. A column knows something from after the customer closed the card. That is cheating.
- **Overlap** is about *repetition*. Two columns saying the same thing. Not cheating, but it muddies explanations.

### 4.1 Leakage — is `salary_lands_in_bank` describing the departure?

When somebody closes their account, their salary stops arriving by definition. If this column was recorded *after* they left, it describes the departure instead of predicting it.

**The method:** assume the worst case is true, work out what the data would look like if it were, then check whether it does. If the column merely recorded leaving, nearly every leaver would show "salary not arriving".

In [ ]:
leavers = df[df[ANSWER] == 1]

still_landing = (leavers.salary_lands_in_bank == 1).sum()
print(f"leavers whose salary STILL landed in the bank: {still_landing:,} of {len(leavers):,}"
      f"  ({still_landing/len(leavers):.0%})")
print()
print("Far from zero, so the column is measuring something real — not the outcome.")

Two supporting checks. If the six months included time *after* people left, we would expect accounts to go dead — months with zero activity.

In [ ]:
zero_purchase = (df[PURCHASES] == 0).any(axis=1).sum()
zero_payment  = (df[PAYMENTS]  == 0).any(axis=1).sum()

print(f"customers with any zero purchase month: {zero_purchase}")
print(f"customers with any zero payment  month: {zero_payment}")
print(f"lowest purchase anywhere in the data  : {df[PURCHASES].min().min():.2f}")
print()
print(f"leavers with more than 10 years as a customer: {(leavers.loyalty_years > 10).sum()}")
print(f"longest relationship among leavers           : {leavers.loyalty_years.max():.1f} years")
print()
print("No dead accounts, and long-standing customers do leave.")
print("So loyalty_years is not secretly counting down to a departure date.")

**Result: no leakage found.** Still to be confirmed by the bank telling us the date each column was recorded — question 2 in the manager doc.

### 4.2 Overlap — are two loan columns saying the same thing?

In [ ]:
crosstab = pd.crosstab(
    df.is_paying_old_loan,
    df.outstanding_loan_balance > 0,
    rownames=["is_paying_old_loan"],
    colnames=["balance > 0"],
)
print(crosstab)
print()

mismatches = ((df.is_paying_old_loan == 1) & (df.outstanding_loan_balance == 0)).sum() \
           + ((df.is_paying_old_loan == 0) & (df.outstanding_loan_balance >  0)).sum()
print(f"mismatches: {mismatches} out of {len(df):,}")
print()
print("Zero mismatches means these describe the same loan.")
print("If they were two different loans, somebody would be repaying one while owing on another.")
print("Keep the balance: it says everything the flag says, plus how much.")

## 5. Comparing customers who closed their card and those who kept it, column by column

This is the whole job: find what is different about the customers who closed their card.

A helper we will reuse — it shows the leaving rate for each value of a column, alongside how many customers are in each group. **The count matters as much as the rate.** A 60% leaving rate on eight people means nothing.

In [ ]:
def leaving_rate(column, data=df, bins=None):
    """Leaving rate per 100 customers, broken down by a column.

    bins: optional list of cut points for a numeric column.
    """
    values = pd.cut(data[column], bins) if bins else data[column]

    out = data.groupby(values, observed=True).agg(
        left_per_100=(ANSWER, lambda s: round(s.mean() * 100, 1)),
        customers=(ANSWER, "size"),
    )
    return out

In [ ]:
for col in ["salary_lands_in_bank", "missed_loan_payment_ever", "has_other_credit_cards",
            "employment_sector", "married", "has_dependents", "gender"]:
    print(f"--- {col} ---")
    print(leaving_rate(col))
    print()

**Gender separates nobody** — the two rates are the same. That is the first of two reasons we remove it. The second is that using gender in decisions about credit or offers is restricted under banking regulation, so even a real gap could not be acted on.

### Numeric columns need bands, not correlations

A correlation only measures how well a *straight line* fits. When a relationship is real but not straight, the number comes out small and the column looks useless. Always look at the bands before dismissing anything.

In [ ]:
print("iscore correlation with leaving:", round(df.iscore.corr(df[ANSWER]), 3), "-- looks weak")
print()
print(leaving_rate("iscore", bins=[0, 580, 650, 720, 900]))
print()
print("A clean staircase. The correlation understated it because the relationship is not a straight line.")

### Salary is the surprise

In [ ]:
print("salary correlation with leaving:", round(df.salary.corr(df[ANSWER]), 4))
print()
print(leaving_rate("salary", bins=[0, 5000, 8000, 12000, 100000]))
print()
print("Essentially flat. Leaving is not about whether people can afford the bank.")
print("It is about whether they are engaged with it. That sentence belongs in the report.")

## 6. The six monthly columns

Individually they look useless. Together they are the strongest thing in the file.

In [ ]:
comparison = pd.DataFrame({
    "stayed": df[df[ANSWER] == 0][PURCHASES].mean(),
    "left":   df[df[ANSWER] == 1][PURCHASES].mean(),
}).round(0)
comparison["gap"] = (comparison["stayed"] - comparison["left"]).round(0)
comparison

Month 1: the two groups are almost identical. Month 6: far apart. They start together and separate.

**The information is in the direction of travel, not in any single month.**

In [ ]:
def plot_monthly_trend(data):
    stayed = data[data[ANSWER] == 0][PURCHASES].mean()
    left   = data[data[ANSWER] == 1][PURCHASES].mean()

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(range(1, 7), stayed.values, marker="o", lw=2.5, color="#1E2761", label="Stayed")
    ax.plot(range(1, 7), left.values,   marker="o", lw=2.5, color="#C0392B", label="Left")
    ax.set_xlabel("Month  (1 = oldest, 6 = most recent)")
    ax.set_ylabel("Average purchases")
    ax.legend(frameon=False)
    ax.grid(axis="y", color="#DDE3EE", lw=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    fig.tight_layout()
    return fig

plot_monthly_trend(df)
plt.show()

## 7. Choosing how to measure the fall

There is no single correct formula. Build several, test which separates the groups best, keep the winner — and if they all perform the same, say so.

**The test:** pick one leaver and one stayer at random. How often does this measure correctly rank the leaver as riskier? 0.50 is a coin flip, 1.00 is always right.

In [ ]:
def separation_score(score, answer):
    """How often this measure ranks a random leaver above a random stayer."""
    ranks = pd.Series(score).rank().values
    n_left   = answer.sum()
    n_stayed = len(answer) - n_left
    auc = (ranks[answer == 1].sum() - n_left * (n_left + 1) / 2) / (n_left * n_stayed)
    return 1 - auc   # falling spend means leaving, so flip


P = df[PURCHASES].values
first3, last3, overall = P[:, :3].mean(1), P[:, 3:].mean(1), P.mean(1)

candidates = {
    "last 3 vs first 3":      (last3 - first3) / first3,
    "month 6 vs month 1":     (P[:, 5] - P[:, 0]) / P[:, 0],
    "month 6 vs the average": (P[:, 5] - overall) / overall,
    "slope of a line":        np.polyfit(np.arange(6), P.T, 1)[0] / overall,
    "month 6 vs best month":  (P[:, 5] - P[:, :5].max(1)) / P[:, :5].max(1),
}

scores = {name: round(separation_score(v, df[ANSWER]), 4) for name, v in candidates.items()}
pd.Series(scores).sort_values(ascending=False).to_frame("separation")

All within about two points of each other — they are measuring the same thing and the choice barely matters. **Reporting that honestly is the finding.**

We use the **slope**, because it uses all six months so one unusual month cannot distort it. We also keep the simple percentage, because "spending fell 30%" is something a marketing person understands and "slope of −87 per month" is not. The model's number and the number you say out loud do not have to be the same.

### Does the steepness matter, or only the direction?

In [ ]:
df["_pct_change"] = (last3 - first3) / first3

print(leaving_rate("_pct_change", bins=[-1, -0.30, -0.20, -0.10, -0.05, 0.05, 10]))
print()
print("A 30% fall is far more dangerous than a 7% fall.")
print("A yes/no 'declining' flag would throw that away, so the trend stays a number.")

## 8. Two columns can mean more together than apart

An **interaction**: two factors that each mean little alone and a great deal in combination. Tree models find these by themselves, which is much of why we chose one.

In [ ]:
df["_declining"] = df["_pct_change"] < -0.05

rates = (pd.crosstab(df.missed_loan_payment_ever, df._declining,
                     df[ANSWER], aggfunc="mean") * 100).round(1)
rates.index.name = "has missed a loan payment"
rates.columns.name = "spending is declining"
print(rates)
print()
print(pd.crosstab(df.missed_loan_payment_ever, df._declining))
print()
print("Neither factor alone explains the bottom-right corner. The combination does.")

## 9. Building the new columns

Everything above was looking. This is building.

Each column below states what it captures. We keep the original file untouched and build into a copy, so we can always go back.

In [ ]:
def build_features(raw):
    """Turn the raw file into the table we will model on.

    Every step here is a decision recorded in docs/PHASE1_ANALYSIS_AND_FEATURES.md.
    """
    d = raw.copy()

    purchases = d[PURCHASES].values
    payments  = d[PAYMENTS].values
    months    = np.arange(6)

    # --- direction of travel: the strongest signal we have ---
    d["purchase_slope"]      = np.polyfit(months, purchases.T, 1)[0] / purchases.mean(1)
    d["payment_slope"]       = np.polyfit(months, payments.T,  1)[0] / payments.mean(1)
    d["purchase_pct_change"] = ((purchases[:, 3:].mean(1) - purchases[:, :3].mean(1))
                                / purchases[:, :3].mean(1))   # for explaining to humans

    # --- level and shape, separate from direction ---
    d["recent_purchases"]    = purchases[:, 3:].mean(1)
    d["purchase_volatility"] = purchases.std(1) / purchases.mean(1)

    # months well below that customer's OWN normal - catches going quiet
    own_normal = purchases.mean(1, keepdims=True)
    d["quiet_months"] = (purchases < 0.6 * own_normal).sum(1)

    # --- are they clearing what they spend, or building up debt? ---
    d["payment_ratio"] = payments.sum(1) / purchases.sum(1)

    # --- how much of their financial life runs through us ---
    d["spend_to_salary"] = purchases.sum(1) / (d.salary * 6)
    d["loan_burden"]     = d.outstanding_loan_balance / d.salary

    # --- household commitments anchor people (came from a rejected hypothesis) ---
    d["responsibility"] = d.married + d.has_dependents

    # --- the more ties to the bank, the harder to leave ---
    d["relationship_depth"] = (d.salary_lands_in_bank
                               + d.had_loan_ever
                               + (1 - d.has_other_credit_cards))

    # --- two customer types needing opposite offers; mainly for the agent ---
    d["churner_type"] = np.where(d.missed_loan_payment_ever == 1, "distressed", "drifting")

    # --- removals, each for a different reason ---
    d = d.drop(columns=[
        "customer_id",         # a name, not a fact about the person
        "gender",              # no signal, and restricted in offer decisions
        "is_paying_old_loan",  # exact duplicate of outstanding_loan_balance > 0
    ])

    return d.drop(columns=[c for c in d.columns if c.startswith("_")])


features = build_features(df)
print(f"{features.shape[0]:,} customers, {features.shape[1]} columns")
print()
new = [c for c in features.columns if c not in df.columns]
print("built:", ", ".join(new))

## 10. Does each new column earn its place?

Building a column is not the same as it being useful. Check every one.

In [ ]:
new_numeric = ["purchase_slope", "payment_slope", "purchase_pct_change",
               "recent_purchases", "purchase_volatility", "quiet_months",
               "payment_ratio", "spend_to_salary", "loan_burden",
               "responsibility", "relationship_depth"]

results = pd.DataFrame({
    "separation": [round(separation_score(features[c], features[ANSWER]), 4)
                   for c in new_numeric],
}, index=new_numeric)

# a column pointing the "wrong" way is still informative - what matters is distance from 0.50
results["strength"] = (results.separation - 0.5).abs().round(4)
results.sort_values("strength", ascending=False)

Anything sitting near 0.50 is not separating the groups and should be questioned. A column that fails here is not automatically deleted — it may still help in combination — but it needs a reason to stay.

In [ ]:
print(leaving_rate("responsibility", data=features))
print()
print(leaving_rate("relationship_depth", data=features))
print()
print(leaving_rate("quiet_months", data=features))

### Are the new columns just copies of each other?

If two columns carry the same information, the model splits its reasoning between them and our explanations end up repeating themselves. Anything above about 0.9 is worth a second look.

In [ ]:
corr = features[new_numeric].corr().abs()
pairs = (corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
             .stack()
             .sort_values(ascending=False))
pairs[pairs > 0.7].round(3).to_frame("correlation")

## 11. Save it

The modelling notebook picks up from here. Nothing above is repeated there.

In [ ]:
import os

OUT = "features.csv" if IN_COLAB else "../data/processed/features.csv"
os.makedirs(os.path.dirname(OUT), exist_ok=True) if not IN_COLAB else None

features.to_csv(OUT, index=False)
print(f"saved {features.shape[0]:,} rows and {features.shape[1]} columns to {OUT}")

# In Colab, download it so it is not lost when the session ends
if IN_COLAB:
    files.download(OUT)

## Where this leaves us

**Done:** the data is understood, two quality problems were checked and settled, and the new columns are built and tested.

**Still open, and none of it blocks the next step:**

- Whether to keep the twelve raw monthly columns now that the trends exist. Test both, keep whichever works better, write down the answer.
- Confirmation from the bank of when each column was recorded.
- What exactly "churned" means, and how long after these six months it was measured.

**Next:** split the data before doing anything else, build a simple baseline, then the real model.